# Lanugage Embedding - Model Preparation

While I have implemented recommended settings in the Model (MixedPrecision, GradientAccumulation, Using GPU, ...) to increase the training speed, none of them had an effect. However, replacing the sentence embedding model with a static version (as recomended by the library to increase performance) has masivly increased the speed from over 1 day per loop to 2 hours per loop (ona  single CPU). Unfortunatly, even when utilizing the onnx backend for the sentence embedder, the speed does not increase further. Using a GPU through Google Colab does not only not even improve the speed, but has a worse performance at 2,5 hours per loop. As the dataloader with the sentence embedding seems to be my current bottleneck, I have decided to attempt to pre-embedd all the products and queries beforehand and save the embedded vectors in a dataset.

In [1]:
import torch
import os

import numpy as np
import pandas as pd

device = torch.device("cpu")
accelerator = "cpu"
data_dir = "esci-data/shopping_queries_dataset"

Products = pd.read_parquet(f"{data_dir}/shopping_queries_dataset_products.parquet").fillna("")
Products.loc[Products.product_color == "Negro (", "product_color"] = "Negro"
Examples = pd.read_parquet(f"{data_dir}/shopping_queries_dataset_examples.parquet").fillna("")


Dataset = pd.merge(Products, Examples, on = ["product_id", "product_locale"]).drop_duplicates()
Dataset.head()

Large_Dataset = Dataset.loc[Dataset.large_version == 1, :].drop(columns = ["small_version", "large_version"])
Large_Dataset.shape

del Dataset
del Products
del Examples

In [2]:
import transformers
bert_name = 'bert-base-uncased'
tokenizer = transformers.BertTokenizer.from_pretrained(bert_name)

In [3]:
tokenizer.vocab_size

30522

In [3]:
columns_for_embedding = ["product_title", "product_brand", "product_color", "product_bullet_point", "product_description", "query"]
columns_with_info = ["product_id", "query_id", "example_id", "split", "esci_label"]
language_filter = Large_Dataset.product_locale == "us"
Large_Dataset = Large_Dataset.loc[language_filter, tuple(columns_for_embedding + columns_with_info)]
Large_Dataset

,product_title,product_brand,product_color,product_bullet_point,product_description,query,product_id,query_id,example_id,split,esci_label
191907,Delta BreezSignature VFB25ACH 80 CFM Exhaust B...,DELTA ELECTRONICS (AMERICAS) LTD.,White,Virtually silent at less than 0.3 sones\nPreci...,,revent 80 cfm,B003O0MNGC,0,13,train,E
191908,Aero Pure AP80RVLW Super Quiet 80 CFM Recessed...,Aero Pure,White,Super quiet 80CFM energy efficient fan virtual...,,revent 80 cfm,B00MARNO5Y,0,12,train,E
191909,Aero Pure AP120H-SL W Slim Fit 120 CFM Bathroo...,Aero Pure,White Finish,"Slim Fit Housing Fits Into 2"" X 6"" Ceiling Joi...",,revent 80 cfm,B011RX6PNO,0,10,train,I
191910,Delta Electronics (Americas) Ltd. RAD80 Delta ...,DELTA ELECTRONICS (AMERICAS) LTD.,With Heater,Quiet operation at 1.5 Sones\nPrecision engine...,,revent 80 cfm,B01MZIK0PI,0,9,train,E
191911,Delta Electronics (Americas) Ltd. GBR80HLED De...,DELTA ELECTRONICS (AMERICAS) LTD.,"With LED Light, Dual Speed & Humidity Sensor",Ultra energy-efficient LED module (11-watt equ...,,revent 80 cfm,B01N5Y6002,0,15,train,E
...,...,...,...,...,...,...,...,...,...,...,...
2506729,"SAYFINE 5 Pack Plain Atheletic Shirts for Men,...",SAYFINE,Black (5 Pack),[VALUE PACK] - Athletic crew neck and flat-loc...,<b>SAYFINE MENS T SHIRTS & GYM T SHIRTS & WORK...,xl tall moisture wicking shirt,B08XX3NL14,114476,2230829,test,E
2506730,GEEK LIGHTING Mens Polo Shirt Sport Casual Sho...,GEEK LIGHTING,012-grey,Material: This mens running shirt is made with...,,xl tall moisture wicking shirt,B08Y8M9Z6L,114476,2230830,test,E
2506731,Alex Vando Mens Golf Shirt Moisture Wicking Qu...,Alex Vando,Blue,Material: Our Mens Golf Shirts are made from 9...,,xl tall moisture wicking shirt,B0936FKSS6,114476,2230831,test,S
2506732,Casei Womens Polo Shirts Golf Shirts Quick Dry...,Casei,White,【Quick Dry & Moisture Wicking】Womens polo shir...,black friday deals 2021 gifts for men gifts fo...,xl tall moisture wicking shirt,B096RXY1VT,114476,2230832,test,E


In [56]:
for c in columns_for_embedding:
    Large_Dataset.loc[:, c] = Large_Dataset.loc[:, c].str.replace("\xad", "")
    Large_Dataset.loc[:, c] = Large_Dataset.loc[:, c].str.replace("\xa0", "")
    Large_Dataset.loc[Large_Dataset.loc[:, c].str.match("^\s*$"), c] = "" 

In [60]:
Large_Dataset[Large_Dataset == ""] = "UNKNOWN"

In [61]:
tensor_size = 32
embedding_func = lambda x: tokenizer.encode_plus(x, add_special_tokens = False, padding = "max_length", max_length = tensor_size, truncation = True)

    
input_ids = lambda x: embedding_func(x)["input_ids"]
attention_masks = lambda x: np.array(embedding_func(x)["attention_mask"])
# embedding_func = lambda x: embedding_model.encode(x,  output_value = "sentence_embedding", truncate_dim = tensor_size, convert_to_tensor = False, batch_size = batch_size * 4, show_progress_bar = True)

In [62]:
multi_column_prep = []
for i, column_name in enumerate(columns_for_embedding):
    if i == 0 or i == len(columns_for_embedding)-1:
        total = tensor_size + 2 # accounting for the adiitonal CLS and SEP
    else:
        total = tensor_size + 1
    for number in range(total):
        multi_column_prep.append((column_name, number))
multi_column_index = pd.MultiIndex.from_tuples(multi_column_prep)

In [63]:
def Dictlist_to_arrays(dict_list):
    input_ids = []
    attention_masks = []
    for d in dict_list:
        input_ids.append(d["input_ids"])
        attention_masks.append(d["attention_mask"])
    input_ids = np.array(input_ids).astype(np.int32)
    attention_masks = np.array(attention_masks).astype(np.int32)
    return input_ids, attention_masks

In [64]:
import torch
from tqdm import tqdm
# batch_size = 1000

CLS_token = np.full(shape = (Large_Dataset.shape[0], 1), fill_value = 101).astype(np.int32)
SEP_token = np.full(shape = (Large_Dataset.shape[0], 1), fill_value = 102).astype(np.int32)
Extra_ones = np.ones(shape = (Large_Dataset.shape[0], 1))

tqdm.pandas(desc = f"Create embeddings for {columns_for_embedding[0]}")
list_of_dicts = Large_Dataset.loc[:, columns_for_embedding[0]].progress_apply(embedding_func).to_list()
Embedded_df, Attention_df = Dictlist_to_arrays(list_of_dicts)
Embedded_df = np.concatenate((CLS_token, Embedded_df, SEP_token), axis = 1)
Attention_df = np.concatenate((Extra_ones, Attention_df, Extra_ones), axis = 1)

for i in range(1, len(columns_for_embedding)):
    tqdm.pandas(desc = f"Create embeddings for {columns_for_embedding[i]}")
    list_of_dicts = Large_Dataset.loc[:, columns_for_embedding[i]].progress_apply(embedding_func).to_list()
    new_Embedded_df, new_Attention_df = Dictlist_to_arrays(list_of_dicts)
    if i == len(columns_for_embedding)-1:
        Embedded_df = np.concatenate((Embedded_df, CLS_token, new_Embedded_df, SEP_token), axis = 1)   
        Attention_df = np.concatenate((Attention_df, Extra_ones, new_Attention_df, Extra_ones), axis = 1)
    else:
        Embedded_df = np.concatenate((Embedded_df, new_Embedded_df, SEP_token), axis = 1)   
        Attention_df = np.concatenate((Attention_df, new_Attention_df, Extra_ones), axis = 1)

Create embeddings for query: 100%|██████████| 1818825/1818825 [03:52<00:00, 7818.11it/s]


In [81]:
Embedded_df = pd.DataFrame(Embedded_df, columns = multi_column_index)
for col in columns_with_info:
    Embedded_df.loc[:, (col, "")] = Large_Dataset.loc[:, col].to_numpy()

Attention_df = pd.DataFrame(Attention_df, columns = multi_column_index)
for col in columns_with_info:
    Attention_df.loc[:, (col, "")] = Large_Dataset.loc[:, col].to_numpy()

In [82]:
Embedded_df

product_title                                                         \
                    0      1      2     3      4      5      6      7      8   
0                 101   7160  21986  2480   5332  16989  11244   1058  26337   
1                 101  18440   5760  9706  17914   2099   2615   2140   2860   
2                 101  18440   5760  9706  12521   2692   2232   1011  22889   
3                 101   7160   8139  1006  10925   1007   5183   1012  10958   
4                 101   7160   8139  1006  10925   1007   5183   1012  16351   
...               ...    ...    ...   ...    ...    ...    ...    ...    ...   
1818820           101   2360  23460  1019   5308   5810   2012  16001  16530   
1818821           101  29294   7497  2273   2015  11037   3797   4368  10017   
1818822           101   4074   3158  3527   2273   2015   5439   3797  14098   
1818823           101   2553   2072  2308   2015  11037  11344   5439  11344   
1818824           101   2273   5439  2836   3797  14098  15536  23177   2146   

                ... query                product_id query_id example_id  \
             9  ...    29 30 31 32   33                                   
0        17788  ...     0  0  0  0  102  B003O0MNGC        0         13   
1         3565  ...     0  0  0  0  102  B00MARNO5Y        0         12   
2         1059  ...     0  0  0  0  102  B011RX6PNO        0         10   
3         2094  ...     0  0  0  0  102  B01MZIK0PI        0          9   
4         2099  ...     0  0  0  0  102  B01N5Y6002        0         15   
...        ...  ...   ... .. .. ..  ...         ...      ...        ...   
1818820  11344  ...     0  0  0  0  102  B08XX3NL14   114476    2230829   
1818821   2460  ...     0  0  0  0  102  B08Y8M9Z6L   114476    2230830   
1818822  15536  ...     0  0  0  0  102  B0936FKSS6   114476    2230831   
1818823   4248  ...     0  0  0  0  102  B096RXY1VT   114476    2230832   
1818824  10353  ...     0  0  0  0  102  B09BW3YVP8   114476    2230828   

         split esci_label  
                           
0        train          E  
1        train          E  
2        train          I  
3        train          E  
4        train          E  
...        ...        ...  
1818820   test          E  
1818821   test          E  
1818822   test          S  
1818823   test          E  
1818824   test          S  

[1818825 rows x 205 columns]

In [83]:
Attention_df

product_title                                               ... query  \
                    0    1    2    3    4    5    6    7    8    9  ...    29   
0                 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  ...   0.0   
1                 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  ...   0.0   
2                 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  ...   0.0   
3                 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  ...   0.0   
4                 1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  ...   0.0   
...               ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...   ...   
1818820           1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  ...   0.0   
1818821           1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  ...   0.0   
1818822           1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  ...   0.0   
1818823           1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  ...   0.0   
1818824           1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  ...   0.0   

                             product_id query_id example_id  split esci_label  
          30   31   32   33                                                    
0        0.0  0.0  0.0  1.0  B003O0MNGC        0         13  train          E  
1        0.0  0.0  0.0  1.0  B00MARNO5Y        0         12  train          E  
2        0.0  0.0  0.0  1.0  B011RX6PNO        0         10  train          I  
3        0.0  0.0  0.0  1.0  B01MZIK0PI        0          9  train          E  
4        0.0  0.0  0.0  1.0  B01N5Y6002        0         15  train          E  
...      ...  ...  ...  ...         ...      ...        ...    ...        ...  
1818820  0.0  0.0  0.0  1.0  B08XX3NL14   114476    2230829   test          E  
1818821  0.0  0.0  0.0  1.0  B08Y8M9Z6L   114476    2230830   test          E  
1818822  0.0  0.0  0.0  1.0  B0936FKSS6   114476    2230831   test          S  
1818823  0.0  0.0  0.0  1.0  B096RXY1VT   114476    2230832   test          E  
1818824  0.0  0.0  0.0  1.0  B09BW3YVP8   114476    2230828   test          S  

[1818825 rows x 205 columns]

In [84]:
Embedded_df.to_parquet("EmbeddedData/WordEmbedding_inputs.parquet")

C:\Users\flash\anaconda3\envs\ML_Pytorch\Lib\site-packages\pandas\io\parquet.py:191: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)


In [85]:
Attention_df.to_parquet("EmbeddedData/WordEmbedding_attention.parquet")

C:\Users\flash\anaconda3\envs\ML_Pytorch\Lib\site-packages\pandas\io\parquet.py:191: UserWarning: The DataFrame has column names of mixed type. They will be converted to strings and not roundtrip correctly.
  table = self.api.Table.from_pandas(df, **from_pandas_kwargs)
